# Training Shelf Detection di Google Colab

Notebook ini hanya menangani training detector YOLO Shelf Detection. Hasil akhirnya adalah `best.pt` yang disimpan secara permanen di Google Drive. Export ONNX, FAISS, API, dan deployment tidak dijalankan di notebook ini.

Jalankan setiap cell secara berurutan dari atas ke bawah. Untuk percobaan pertama, gunakan 5 epoch. Setelah seluruh pipeline berhasil, ubah menjadi 50-100 epoch.

## 1. Pilih GPU Colab

Sebelum menjalankan notebook, buka **Runtime > Change runtime type**, lalu pilih **T4 GPU** atau GPU lain yang tersedia. Training YOLOv8m dengan CPU akan sangat lambat.

## 2. Konfigurasi utama

Cell berikut mengumpulkan seluruh nilai yang mungkin perlu diubah:

- `DATASET_NAME`: descriptor resmi SKU-110K dari Ultralytics yang akan diunduh otomatis.
- `EPOCHS`: jumlah putaran training. Gunakan 5 untuk smoke test.
- `BATCH`: jumlah gambar per batch. Turunkan ke 4 atau 2 jika GPU kehabisan memori.
- `USE_WANDB`: aktifkan dashboard online W&B jika API key sudah disimpan di Colab Secrets.
- `RESUME_CHECKPOINT`: isi dengan path `last.pt` bila ingin melanjutkan training yang terputus.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone

REPO_URL = "https://github.com/kholiqdev/shelf-detection.git"
REPO_DIR = Path("/content/shelf-detection")
DRIVE_ROOT = Path("/content/drive/MyDrive/shelf-detection")
DATASET_NAME = "SKU-110K.yaml"
RUNS_DIR = DRIVE_ROOT / "runs"
MODELS_DIR = DRIVE_ROOT / "models"

EPOCHS = 5
BATCH = 8
IMAGE_SIZE = 640
WORKERS = 2
RUN_NAME = "shelf-detection-colab-v1"
USE_WANDB = False
RESUME_CHECKPOINT = ""

print(f"Dataset      : {DATASET_NAME}")
print(f"Epoch / batch: {EPOCHS} / {BATCH}")
print(f"Run name     : {RUN_NAME}")

## 3. Mount Google Drive

`drive.mount()` meminta izin agar Colab dapat menulis checkpoint. Direktori `runs`, `models`, dan `wandb` dibuat di Drive supaya hasil tidak hilang ketika runtime Colab berhenti. Dataset diproses di storage lokal Colab agar training lebih cepat.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
for directory in (DRIVE_ROOT, RUNS_DIR, MODELS_DIR, DRIVE_ROOT / "wandb"):
    directory.mkdir(parents=True, exist_ok=True)

print("Google Drive siap untuk menyimpan hasil training.")

## 4. Periksa GPU

PyTorch harus mendeteksi CUDA. `nvidia-smi` menampilkan tipe GPU, VRAM, driver, dan penggunaan memori. Notebook dihentikan lebih awal jika GPU belum aktif agar training tidak tanpa sengaja berjalan di CPU.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU CUDA tidak tersedia. Pilih Runtime > Change runtime type > T4 GPU."
    )

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"CUDA    : {torch.version.cuda}")
subprocess.run(["nvidia-smi"], check=False)

## 5. Clone atau perbarui repository

Repository terbaru adalah `git@github.com:kholiqdev/shelf-detection.git`. Colab menggunakan padanan HTTPS publik agar tidak memerlukan SSH key, lalu clone disimpan di `/content/shelf-detection`. Jika cell dijalankan ulang, `git pull --ff-only` hanya mengambil update yang dapat diterapkan tanpa membuat merge otomatis.

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True
    )

os.chdir(REPO_DIR)
git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Repository : {REPO_DIR}")
print(f"Git commit : {git_commit}")

## 6. Instal dependency khusus training

Notebook hanya memasang `ultralytics` dan `wandb`. Dependency CLIP, FAISS, FastAPI, dan ONNX tidak diperlukan untuk training detector sehingga sengaja tidak dipasang. `pip` menggunakan PyTorch CUDA yang sudah tersedia di image Colab.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics>=8.4.70",
        "wandb>=0.18.0",
    ],
    check=True,
)

import ultralytics
import wandb
from ultralytics import YOLO

print(f"Ultralytics: {ultralytics.__version__}")
print(f"W&B        : {wandb.__version__}")

## 7. Atur W&B

Secara default W&B berjalan offline dan log-nya disimpan di Drive. Untuk dashboard online, buat secret `WANDB_API_KEY` melalui ikon kunci di sidebar Colab lalu ubah `USE_WANDB = True`. API key tidak ditulis langsung di notebook.

In [ ]:
os.environ["WANDB_DIR"] = str(DRIVE_ROOT / "wandb")

if USE_WANDB:
    from google.colab import userdata

    api_key = userdata.get("WANDB_API_KEY")
    if not api_key:
        raise RuntimeError("Secret WANDB_API_KEY belum dibuat di Colab.")
    os.environ["WANDB_API_KEY"] = api_key
    os.environ["WANDB_MODE"] = "online"
    print("W&B online aktif.")
else:
    os.environ["WANDB_MODE"] = "offline"
    print(f"W&B offline. Log disimpan di {os.environ['WANDB_DIR']}")

## 8. Unduh dan periksa SKU-110K resmi

Ultralytics menyediakan `SKU-110K.yaml` dengan script download dan konversi anotasi resmi. Cell ini mengunduh sekitar 13.6 GB ke storage lokal Colab pada penggunaan pertama, lalu memastikan split train, val, dan test tersedia dan tidak kosong.

In [ ]:
from ultralytics.data.utils import check_det_dataset

dataset_info = check_det_dataset(DATASET_NAME)
dataset_root = Path(dataset_info["path"])
dataset_counts = {}

for split in ("train", "val", "test"):
    split_value = dataset_info.get(split)
    if not split_value:
        raise RuntimeError(f"Split {split} tidak tersedia pada {DATASET_NAME}.")

    split_sources = split_value if isinstance(split_value, list) else [split_value]
    image_count = 0
    for split_source in split_sources:
        split_path = Path(split_source)
        if split_path.is_file():
            image_count += sum(
                1
                for line in split_path.read_text().splitlines()
                if line.strip()
            )
        elif split_path.is_dir():
            image_count += sum(
                1
                for path in split_path.rglob("*")
                if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
            )
        else:
            raise FileNotFoundError(f"Sumber split {split} tidak ditemukan: {split_path}")

    if image_count == 0:
        raise RuntimeError(f"Split {split} kosong.")
    dataset_counts[split] = {"images": image_count}
    print(f"{split:>5}: {image_count:>6} images")

print(f"Dataset root: {dataset_root}")

## 9. Buat konfigurasi training khusus Colab

Konfigurasi training dibuat langsung di notebook agar tidak bergantung pada file lain di repository. Output `project` diarahkan ke Google Drive, sedangkan dataset tetap berada di storage lokal. `device: 0` berarti GPU CUDA pertama.

In [ ]:
training_config = {
    "epochs": EPOCHS,
    "batch": BATCH,
    "imgsz": IMAGE_SIZE,
    "device": 0,
    "workers": WORKERS,
    "project": str(RUNS_DIR),
    "name": RUN_NAME,
    "exist_ok": False,
}
print(json.dumps({"data": DATASET_NAME, **training_config}, indent=2))

## 10. Jalankan training

Untuk training baru, cell memanggil API Ultralytics langsung dengan konfigurasi Colab dan descriptor resmi SKU-110K. Ultralytics juga akan mengunduh pretrained `yolov8m.pt` pada run pertama.

Jika `RESUME_CHECKPOINT` diisi dengan path `last.pt`, Ultralytics melanjutkan epoch dan state optimizer dari checkpoint tersebut. Jangan mengisi `RESUME_CHECKPOINT` untuk run baru.

In [ ]:
if RESUME_CHECKPOINT:
    checkpoint_path = Path(RESUME_CHECKPOINT)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Checkpoint tidak ditemukan: {checkpoint_path}")
    print(f"Melanjutkan training dari {checkpoint_path}")
    YOLO(str(checkpoint_path)).train(resume=True)
else:
    print("Memulai training baru dengan Ultralytics.")
    YOLO("yolov8m.pt").train(data=DATASET_NAME, **training_config)

## 11. Simpan model dan metadata

Notebook mencari `best.pt` terbaru di folder run, lalu menyalinnya ke dua lokasi:

- `models/<RUN_NAME>/best.pt` untuk menyimpan versi run.
- `models/best.pt` sebagai path sederhana untuk notebook POC.

SHA-256, commit Git, parameter training, dan statistik dataset disimpan sebagai `metadata.json`. Checksum membantu memastikan file model tidak berubah atau rusak saat dipindahkan.

In [ ]:
best_candidates = list(RUNS_DIR.glob("*/weights/best.pt"))
if not best_candidates:
    raise FileNotFoundError(f"best.pt tidak ditemukan di {RUNS_DIR}")

best_source = max(best_candidates, key=lambda path: path.stat().st_mtime)
version_dir = MODELS_DIR / RUN_NAME
version_dir.mkdir(parents=True, exist_ok=True)
versioned_model = version_dir / "best.pt"
latest_model = MODELS_DIR / "best.pt"
shutil.copy2(best_source, versioned_model)
shutil.copy2(best_source, latest_model)
sha256 = hashlib.sha256(versioned_model.read_bytes()).hexdigest()
metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_name": RUN_NAME,
    "git_commit": git_commit,
    "model_source": str(best_source),
    "model_path": str(versioned_model),
    "sha256": sha256,
    "training_config": {"data": DATASET_NAME, **training_config},
    "dataset_counts": dataset_counts,
}
metadata_path = version_dir / "metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2))

print(f"Model versi : {versioned_model}")
print(f"Model latest: {latest_model}")
print(f"Metadata    : {metadata_path}")
print(f"SHA-256     : {sha256}")

## Selesai

Training selesai ketika cell sebelumnya menampilkan path model dan SHA-256. File yang akan dipakai oleh notebook POC adalah:

```text
/content/drive/MyDrive/shelf-detection/models/best.pt
```

Untuk training penuh, ubah `EPOCHS` menjadi 50-100 lalu jalankan ulang notebook dari atas. Gunakan nama run baru agar hasil smoke test tidak tertukar dengan model final.